# 0 · Dicionário de dados

**CRISP-DM fase 2.2 — descrição dos dados** · Meridiano Imobiliária

---

Este notebook monta o **dicionário de dados da base bruta**: para cada uma das
82 colunas de `data/raw/ames_housing.parquet`, registra o nome original, uma
descrição de negócio, a faixa (ou os extremos) dos valores observados e o tipo
de variável.

Três ajustes em relação ao script de referência, todos anotados no ponto em que
aparecem:

1. a leitura passa a consumir o **arquivo bruto em `data/raw/`**, não um CSV já
   processado — o dicionário descreve o dado como ele chega;
2. a checagem de tipo usa `pandas.api.types` em vez de comparar `dtype` com a
   string `'object'`: no pandas 3 as colunas de texto têm dtype `str`, e a
   comparação literal deixaria todas elas de fora;
3. o ramo **Ordinal**, que no script original nunca era alcançado, passa a ser
   decidido pelas escalas declaradas em `src/data/dicionario.py` — a fonte única
   da verdade do projeto sobre quais colunas têm ordem semântica.

In [ ]:
# importações
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

from src import config
from src.data.dicionario import ESCALAS_ORDINAIS

print(f"Projeto em {RAIZ}")

## Carregando a base de dados

A fonte é o arquivo bruto baixado do endereço oficial do artigo de De Cock
(2011). Existem duas cópias equivalentes em `data/raw/`: o `.txt` original,
separado por tabulação, e um `.parquet` gravado na ingestão — mesma tabela, mas
com os tipos preservados e leitura ~50× mais rápida. O parquet é a leitura
preferencial; o `.txt` fica como plano B caso a ingestão ainda não tenha rodado.

In [ ]:
# Carregando base de dados
if config.ARQUIVO_BRUTO_PARQUET.exists():
    df = pd.read_parquet(config.ARQUIVO_BRUTO_PARQUET)
    origem = config.ARQUIVO_BRUTO_PARQUET
else:
    # O arquivo publicado pelo autor é delimitado por tabulação, não por `;`.
    df = pd.read_csv(config.ARQUIVO_BRUTO, sep="\t")
    origem = config.ARQUIVO_BRUTO

print(f"{origem.relative_to(RAIZ)} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

## Varredura das colunas

Percorremos coluna a coluna classificando o tipo e coletando os valores de
referência:

| dtype | tipo atribuído | valores registrados |
|---|---|---|
| inteiro | Discreta | mínimo e máximo |
| decimal | Contínua | mínimo e máximo |
| texto | Nominal | primeira e última categoria observada |

A exceção é a coluna que aparece em `ESCALAS_ORDINAIS`: ela é texto, mas tem
ordem (`Ex > Gd > TA > Fa > Po`). Nesses casos o tipo vira **Ordinal** e os
valores registrados são os extremos da escala declarada — o pior e o melhor
nível —, não a ordem de aparição no arquivo.

In [ ]:
# lista de dicionários para armazenar informações sobre as colunas
col_info = []


# Converte escalar numpy em escalar Python. Sem isso o CSV sai com
# `[np.int64(12789), np.int64(755000)]` em vez de `[12789, 755000]` — é só o
# `repr` do numpy 2, mas o dicionário é lido por gente, não por máquina.
def escalar(valor):
    return valor.item() if hasattr(valor, "item") else valor

# percorrendo as colunas do dataframe
for col in df.columns:
    # variável
    variable = col
    # tipo
    if col in ESCALAS_ORDINAIS:
        # Ordinal tem precedência sobre Nominal: a coluna é texto, mas a escala
        # declarada em src/data/dicionario.py lhe dá ordem.
        col_type = "Ordinal"
        escala = ESCALAS_ORDINAIS[col]
        niveis = sorted(escala, key=escala.get)
        values = [niveis[0], niveis[-1]]
    elif pd.api.types.is_integer_dtype(df[col]):
        col_type = "Discreta"
        values = [escalar(df[col].min()), escalar(df[col].max())]
    elif pd.api.types.is_float_dtype(df[col]):
        col_type = "Contínua"
        values = [escalar(df[col].min()), escalar(df[col].max())]
    elif pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
        col_type = "Nominal"
        values = np.array(df[col].dropna().unique())
        values = [values[0], values[-1]]
    else:
        col_type = "Não classificada"
        values = []
    # descrição (preenchida logo abaixo, a partir do mapa de negócio)
    description = "Descrição da coluna"

    # adicionando informações à lista de dicionários
    col_info.append(
        {
            "variavel": variable,
            "descricao": description,
            "valores": values,
            "Tipo": col_type,
        }
    )

In [ ]:
# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

print(f"{len(col_info_df)} colunas descritas")
col_info_df["Tipo"].value_counts()

## Descrições de negócio

O script de referência usa uma **lista posicional** de descrições, casada com a
ordem das colunas. Aqui a mesma informação vira **dicionário**: com 82 colunas,
uma lista posicional quebra em silêncio se a ordem do arquivo mudar — e a
descrição errada colada na variável errada é pior que descrição nenhuma. A lista
continua existindo, mas é derivada do mapa, na ordem em que as colunas
efetivamente apareceram.

In [ ]:
# Descrições de negócio, por nome original da coluna
DESCRICOES = {
    "Order": "Número de ordem do registro no arquivo original",
    "PID": "Identificador cadastral do lote na prefeitura (Parcel ID)",
    "MS SubClass": "Código do tipo de imóvel envolvido na venda (idade, pavimentos e estilo)",
    "MS Zoning": "Classificação geral de zoneamento urbano do lote",
    "Lot Frontage": "Testada: pés lineares de rua conectados ao lote",
    "Lot Area": "Área do terreno, em pés quadrados",
    "Street": "Tipo de pavimentação da via de acesso",
    "Alley": "Tipo de pavimentação da viela de acesso (NA = sem viela)",
    "Lot Shape": "Regularidade do formato do lote",
    "Land Contour": "Relevo do terreno (plano, encosta, depressão)",
    "Utilities": "Infraestrutura urbana disponível no lote",
    "Lot Config": "Configuração do lote (interno, esquina, fundo de rua)",
    "Land Slope": "Declividade do terreno",
    "Neighborhood": "Bairro de Ames em que o imóvel se localiza",
    "Condition 1": "Proximidade de via principal, ferrovia ou área de interesse",
    "Condition 2": "Segunda proximidade relevante, quando há mais de uma",
    "Bldg Type": "Tipo de construção (isolada, geminada, sobreposta)",
    "House Style": "Estilo da residência e número de pavimentos",
    "Overall Qual": "Qualidade geral do material e do acabamento (1 a 10)",
    "Overall Cond": "Estado geral de conservação (1 a 10)",
    "Year Built": "Ano da construção original",
    "Year Remod/Add": "Ano da última reforma ou ampliação (igual à construção se nunca houve)",
    "Roof Style": "Estilo do telhado",
    "Roof Matl": "Material do telhado",
    "Exterior 1st": "Revestimento externo predominante",
    "Exterior 2nd": "Segundo revestimento externo, quando há mais de um material",
    "Mas Vnr Type": "Tipo de alvenaria aparente (None = sem revestimento aparente)",
    "Mas Vnr Area": "Área de alvenaria aparente, em pés quadrados",
    "Exter Qual": "Qualidade do material de revestimento externo",
    "Exter Cond": "Estado de conservação do revestimento externo",
    "Foundation": "Tipo de fundação",
    "Bsmt Qual": "Altura e qualidade do porão (NA = sem porão)",
    "Bsmt Cond": "Estado de conservação do porão (NA = sem porão)",
    "Bsmt Exposure": "Exposição do porão à luz externa ou saída para o jardim",
    "BsmtFin Type 1": "Qualidade do acabamento da primeira área acabada do porão",
    "BsmtFin SF 1": "Área acabada tipo 1 do porão, em pés quadrados",
    "BsmtFin Type 2": "Qualidade do acabamento da segunda área acabada do porão",
    "BsmtFin SF 2": "Área acabada tipo 2 do porão, em pés quadrados",
    "Bsmt Unf SF": "Área do porão sem acabamento, em pés quadrados",
    "Total Bsmt SF": "Área total do porão, em pés quadrados (0 = sem porão)",
    "Heating": "Tipo de sistema de aquecimento",
    "Heating QC": "Qualidade e condição do sistema de aquecimento",
    "Central Air": "Possui ar-condicionado central (Y/N)",
    "Electrical": "Tipo de sistema elétrico",
    "1st Flr SF": "Área do pavimento térreo, em pés quadrados",
    "2nd Flr SF": "Área do pavimento superior, em pés quadrados (0 = pavimento único)",
    "Low Qual Fin SF": "Área acabada de baixa qualidade, somando todos os pavimentos",
    "Gr Liv Area": "Área habitável acima do solo, em pés quadrados",
    "Bsmt Full Bath": "Banheiros completos no porão",
    "Bsmt Half Bath": "Lavabos no porão",
    "Full Bath": "Banheiros completos acima do solo",
    "Half Bath": "Lavabos acima do solo",
    "Bedroom AbvGr": "Dormitórios acima do solo (não inclui os do porão)",
    "Kitchen AbvGr": "Cozinhas acima do solo",
    "Kitchen Qual": "Qualidade da cozinha",
    "TotRms AbvGrd": "Total de cômodos acima do solo, exceto banheiros",
    "Functional": "Funcionalidade da residência, com dedução por limitações de uso",
    "Fireplaces": "Número de lareiras",
    "Fireplace Qu": "Qualidade da lareira (NA = sem lareira)",
    "Garage Type": "Posição da garagem em relação à casa",
    "Garage Yr Blt": "Ano de construção da garagem",
    "Garage Finish": "Acabamento interno da garagem",
    "Garage Cars": "Capacidade da garagem, em número de vagas",
    "Garage Area": "Área da garagem, em pés quadrados",
    "Garage Qual": "Qualidade da garagem (NA = sem garagem)",
    "Garage Cond": "Estado de conservação da garagem (NA = sem garagem)",
    "Paved Drive": "Pavimentação do acesso de veículos",
    "Wood Deck SF": "Área de deck de madeira, em pés quadrados",
    "Open Porch SF": "Área de varanda aberta, em pés quadrados",
    "Enclosed Porch": "Área de varanda fechada, em pés quadrados",
    "3Ssn Porch": "Área de varanda de três estações, em pés quadrados",
    "Screen Porch": "Área de varanda com tela, em pés quadrados",
    "Pool Area": "Área da piscina, em pés quadrados (0 = sem piscina)",
    "Pool QC": "Qualidade da piscina (NA = sem piscina)",
    "Fence": "Tipo e qualidade da cerca (NA = sem cerca)",
    "Misc Feature": "Item extra não coberto pelas demais colunas (galpão, quadra, elevador)",
    "Misc Val": "Valor do item extra, em dólares",
    "Mo Sold": "Mês da venda (1 a 12)",
    "Yr Sold": "Ano da venda (2006 a 2010)",
    "Sale Type": "Modalidade da venda",
    "Sale Condition": "Condição da venda (normal, parcial, familiar, execução)",
    "SalePrice": "Preço de venda em dólares — rótulo reservado, fora da modelagem",
}

# Lista na ordem em que as colunas apareceram no arquivo. O KeyError aqui é
# proposital: coluna nova na base bruta tem de ser descrita, não silenciada.
descricoes = [DESCRICOES[variavel] for variavel in col_info_df["variavel"]]

assert len(descricoes) == len(col_info_df)

In [ ]:
# Adicionando descrição para cada uma das variáveis selecionadas
for i in range(0, len(col_info_df)):
    col_info_df.loc[i, "descricao"] = descricoes[i]

col_info_df

## Salvando o dicionário de dados

O arquivo vai para `data/processed/dicionario.csv`, separado por `;` e sem
índice — o mesmo formato consumido pelos demais notebooks e pela aplicação.

> As descrições da **camada curada** (nomes em `snake_case` português, já
> derivados e agregados) vivem em `src/data/dicionario.py`. Este CSV descreve a
> base **bruta**: é o retrato de entrada, anterior a qualquer transformação.

In [ ]:
# Salvando dicionário de dados
config.garantir_diretorios()
destino = config.DATA_PROCESSED / "dicionario.csv"
col_info_df.to_csv(destino, index=False, sep=";")

print(f"Dicionário gravado em {destino.relative_to(RAIZ)} ({destino.stat().st_size:,} bytes)")
pd.read_csv(destino, sep=";").head(10)